## Lab2 : LCEL Deep Dive + Structured Output with Azure OpenAI

In [2]:
# Basic LCEL Parser 
# load all packages
# Setp 1: 
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os
load_dotenv()  # Load environment variables from .env file

True

In [10]:
# Step 2: 
# azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
# azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")
model = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version = os.getenv("AZURE_OPENAI_API_VERSION")
)

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in one paragraph."
    )

In [ ]:
chain = prompt | model | StrOutputParser()

response = chain.invoke({"topic": "LangChain"})
print(response)

LangChain is a framework designed to simplify building applications powered by large language models by connecting them with external data, tools, and reasoning processes. It provides abstractions for handling prompts, chaining model calls, managing context, and integrating with sources like databases, APIs, and document stores. Instead of treating an LLM as a single standalone text generator, LangChain helps developers create more complex pipelines where the model can retrieve information, call functions, reason step-by-step, and interact with the outside world.


In [12]:
chain =(
    prompt | model | StrOutputParser()

)
print(chain.invoke({"topic": "AI Agent"}))

An AI agent is a software system designed to perceive its environment, make decisions, and take actions to achieve specific goals with minimal or no human intervention. It uses techniques like machine learning, reasoning, and planning to interpret data, decide what to do next, and interact with digital or real-world environments. Depending on its design, an AI agent can operate autonomously, learn from experience, and adapt its behavior over time, making it useful for tasks like automation, problem solving, and personalized assistance.


In [13]:
# step 3: Structure output parser
prompt = ChatPromptTemplate.from_template("""
You are an AI that outputs JSON only.
Return response in this format:
{{
 "topic":{topic},
 "summary": "shorrt summary of the topic"
 "difficulty": "easy, medium, hard"                                                                                                                   
}}
""")

chain = prompt | model | StrOutputParser()
response = chain.invoke({"topic": "LangChain"})
print(response)

{
 "topic": "LangChain",
 "summary": "LangChain is a framework that helps developers build applications powered by large language models by providing tools for chaining model calls, integrating external data, and managing prompts.",
 "difficulty": "easy"
}


In [14]:
# Step 4 : Using JSON Output Parser
from langchain_core.output_parsers import JsonOutputParser
chain = prompt | model | JsonOutputParser()
response = chain.invoke({"topic": "LangChain"})
print(response)

{'topic': 'LangChain', 'summary': 'LangChain is a framework for building applications powered by large language models, making it easier to connect models with tools, data sources, and chains of reasoning.', 'difficulty': 'easy'}


In [ ]:
from pydantic import BaseModel,Field
class TopicInfo(BaseModel):
    topic: str = Field(description="Topic name")
    summary: str = Field(description="A short summary of the topic")
    difficulty: str = Field(description="Difficulty level")
parser= JsonOutputParser(output_model=TopicInfo)
prompt = ChatPromptTemplate.from_template("""
{format_instructions}
Explain the topic: {topic}
""").partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

response = chain.invoke({"topic": "LangChain"})
print(response)



{'topic': 'LangChain', 'explanation': 'LangChain is a software framework designed to make it easier for developers to build applications powered by large language models (LLMs). Instead of interacting with an LLM in isolation, LangChain helps connect the model to external data sources, tools, and chains of logic. This allows developers to create more advanced AI applications such as chatbots, retrieval‑augmented generation systems, autonomous agents, and workflow automations. Core features include prompt management, memory for storing conversation context, integrations with vector databases for semantic search, and tools for chaining multiple model calls or functions together in a structured way. The framework has become popular because it simplifies building complex AI-driven systems that rely on both language models and real-world data or services.'}
